# Lab 6 · Troubleshooting and optimization: skew, joins and file layout

Create a skewed key and fix it with salting, prove a broadcast join in the query plan, turn many small files into a few with OPTIMIZE, switch a table to liquid clustering, and read job history like an engineer on call.

**Exam:** Databricks Certified Data Engineer Associate · **Domain:** Troubleshooting, Monitoring, and Optimization (10% of the exam) · **Time:** about 30 minutes

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end. Everything is created in the `workspace.dea_lab` schema, away from your other work.

From the Learning Hub certification guide: [Data Engineer Associate](https://www.datapreplab.com/hub#certifications/data-engineer-associate) · Independent study resource, not affiliated with or endorsed by Databricks.

## Where to look on serverless

On classic compute you diagnose Spark work in the **Spark UI** (stages, task summary, spill). Serverless compute, which this notebook uses, shows the **query profile** instead: after a cell runs, click **See performance** under it and open the query. The exam asks about both, and the signals are the same: one task far slower than the rest is skew, large spill means partitions too big for memory, and a huge exchange before a join suggests broadcasting the small side.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.dea_lab;
USE workspace.dea_lab;

## Part 1 · A skewed key

**Predict:** what share of rows share one key?

In [ ]:
from pyspark.sql import functions as F

events = (spark.range(0, 3_000_000)
    .withColumn("customer_id", F.when(F.col("id") % 10 < 9, F.lit(0)).otherwise(F.col("id") % 5000))
    .withColumn("amount", (F.col("id") % 100).cast("double")))
display(events.groupBy("customer_id").count().orderBy(F.desc("count")).limit(5))

Customer 0 owns 90% of the rows. In a shuffle, all rows for one key go to one task, so that task does most of the work while the others wait.

## Part 2 · Salting the key

Split the hot key into 16 sub-keys, aggregate, then combine the partial results.

In [ ]:
salted = (events.withColumn("salt", (F.rand(seed=7) * 16).cast("int"))
    .groupBy("customer_id", "salt").agg(F.sum("amount").alias("partial"))
    .groupBy("customer_id").agg(F.sum("partial").alias("revenue")))
plain = events.groupBy("customer_id").agg(F.sum("amount").alias("revenue"))

check = plain.join(salted, "customer_id").where(plain.revenue != salted.revenue).count()
print("Customers where the two answers differ:", check)

Open **See performance** for the cell and compare the aggregation stages. At this small size both are quick; the point is the pattern. For a skewed **join**, Adaptive Query Execution (AQE) splits large partitions automatically. A skewed **groupBy** needs a fix like this one.

## Part 3 · A broadcast join in the plan

**Predict:** which join strategy appears in the physical plan?

In [ ]:
stores = spark.createDataFrame([(i, f"store-{i}") for i in range(50)], "store_id INT, name STRING")
orders = spark.range(0, 1_000_000).withColumn("store_id", (F.col("id") % 50).cast("int"))
orders.join(F.broadcast(stores), "store_id").explain()

`BroadcastHashJoin` sends the small table to every task, so the large side is never shuffled.

## Part 4 · Many small files, then OPTIMIZE

In [ ]:
%sql
DROP TABLE IF EXISTS ops_events;
CREATE TABLE ops_events (event_id BIGINT, customer_id BIGINT, event_date DATE);

In [ ]:
for day in range(20):                    # twenty tiny appends, like a stream that writes often
    spark.sql(f"INSERT INTO ops_events SELECT id + {day * 100}, id % 7, date_add('2026-11-01', {day}) FROM range(100)")

**Predict:** how many files before and after OPTIMIZE?

In [ ]:
def files(table="ops_events"):
    return spark.sql(f"DESCRIBE DETAIL {table}").first()["numFiles"]

print("Files before OPTIMIZE:", files())
spark.sql("OPTIMIZE ops_events")
print("Files after OPTIMIZE:", files())
display(spark.sql("DESCRIBE HISTORY ops_events")
        .where("operation = 'OPTIMIZE'")
        .select("version", "operation", "operationMetrics.numRemovedFiles", "operationMetrics.numAddedFiles"))

OPTIMIZE compacts small files into larger ones. The old files stay on storage (for time travel) until VACUUM removes files the table no longer references and that are older than the retention period, 7 days by default.

In [ ]:
%sql
VACUUM ops_events;   -- removes nothing yet: the replaced files are younger than 7 days

## Part 5 · Liquid clustering

Liquid clustering keeps rows with similar key values in the same files, and you can change the keys later without rewriting the table by hand.

In [ ]:
%sql
ALTER TABLE ops_events CLUSTER BY (customer_id);
OPTIMIZE ops_events;

In [ ]:
print("Clustered by:", spark.sql("DESCRIBE DETAIL ops_events").first()["clusteringColumns"])
spark.sql("ALTER TABLE ops_events CLUSTER BY (customer_id, event_date)")   # new keys apply to future clustering
print("Clustered by:", spark.sql("DESCRIBE DETAIL ops_events").first()["clusteringColumns"])

On Unity Catalog managed tables, **predictive optimization** can run OPTIMIZE and VACUUM for you, so in production you rarely schedule them yourself.

## Part 6 · Read job history (UI, no code)

Open **Jobs & Pipelines** and the `dea_lab_daily_orders` job created by the **04-jobs** notebook. Part 7 of that notebook deletes the job, so if you already ran it, run Parts 1 to 4 of 04-jobs again first.

1. The **Runs** tab shows a bar per run: its height is the duration. One tall bar is noise; a steady climb is a trend worth fixing.
2. The matrix shows each task per run, so you can see which upstream task failed and blocked the rest.
3. Click a failed task to read its error and output, then **Repair run**.

## Expected results

```text
Part 1: customer 0 has 2,700,000 rows (90%); the other 500 customers have 600 rows each.
Part 2: "Customers where the two answers differ: 0" (salting changes the work, not the answer).
Part 3: the plan contains BroadcastHashJoin (and BroadcastExchange on the stores side).
Part 4: before OPTIMIZE about 20 files (one or more per insert); after OPTIMIZE 1 file (or very few).
        The OPTIMIZE row shows about 20 removed files and 1 added.
Part 5: clusteringColumns is [customer_id], then [customer_id, event_date].
```

## Exam takeaways

- More executors do not fix one skewed task; salting or AQE (for joins) does.
- Spill to disk on most tasks: partitions are too large for memory.
- OPTIMIZE compacts; VACUUM deletes unreferenced files older than the retention period.
- Liquid clustering keys can change later; Hive-style partition columns cannot without a rewrite.